# Session 13 · The Confusion Matrix

**Machine Learning Foundations · Sanketana School of Code**

We close the classification module by putting everything into **one small grid**. Session 12 named the two mistakes; today we lay all four outcomes — fraud **caught**, fraud **missed**, honest card **falsely flagged**, honest card **cleared** — in a 2×2 table: the **confusion matrix**. Then we slide the threshold and watch the grid move — which is where the module's **ethics** reaches its peak.

By the end of this notebook you will be able to:

- draw and **label** a confusion matrix in plain language
- read **precision** (a column) and **recall** (a row) straight off the grid
- move the **threshold** and watch every cell shift — the precision/recall trade-off
- choose a threshold for a **stated policy** and defend it as a decision about **people**

## Warm-up · Last session's homework

Your coach will walk through Session 12's accuracy-trap homework (about 10 minutes): the 96%/0-recall baseline, and everyone's ethics stand on which mistake is worse. Today we assemble those four outcomes into one object and make it move.

## Step 1 · Fit the fraud model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
X = fraud.drop(columns=["is_fraud", "transaction_id"]).values
y = fraud["is_fraud"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)

# Probabilities once; we re-cut them at different thresholds all session.
proba = model.predict_proba(scaler.transform(X_test))[:, 1]
print("test transactions:", len(y_test), " | real frauds:", int(y_test.sum()))

## Step 2 · Draw the grid at the default 0.5 cut

Every transaction lands in exactly one of four cells. The helper draws them with plain-language names.

In [ ]:
def draw_matrix(y_true, pred, ax, title):
    """Confusion matrix, actual-fraud row first, cells named in plain language."""
    # labels=[1, 0] puts 'fraud' first on both axes, matching the board grid.
    cm = confusion_matrix(y_true, pred, labels=[1, 0])
    names = [["caught\n(TP)", "missed\n(FN)"],
             ["false alarm\n(FP)", "cleared\n(TN)"]]
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{names[i][j]}\n{cm[i, j]}", ha="center", va="center",
                    fontsize=11, color="black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["fraud", "legit"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["fraud", "legit"])
    ax.set_xlabel("PREDICTED"); ax.set_ylabel("ACTUAL")
    ax.set_title(title)

In [ ]:
pred_05 = (proba >= 0.5).astype(int)
fig, ax = plt.subplots(figsize=(5, 5))
draw_matrix(y_test, pred_05, ax, "Fraud model at threshold = 0.5")
plt.tight_layout(); plt.show()

tn = int(((pred_05==0)&(y_test==0)).sum()); fp = int(((pred_05==1)&(y_test==0)).sum())
fn = int(((pred_05==0)&(y_test==1)).sum()); tp = int(((pred_05==1)&(y_test==1)).sum())
print(f"caught (TP)={tp}  missed (FN)={fn}  false alarm (FP)={fp}  cleared (TN)={tn}")

## Step 3 · Read precision and recall *off* the grid

No formulas to memorise — just trace the grid:

- **recall** = the *"actually fraud" row*: caught ÷ all real fraud = TP / (TP + FN)
- **precision** = the *"said fraud" column*: caught ÷ all alarms = TP / (TP + FP)

In [ ]:
recall_by_hand = tp / (tp + fn)
precision_by_hand = tp / (tp + fp)
print("recall  (row)    =", round(recall_by_hand, 2), " matches recall_score? ",
      round(recall_score(y_test, pred_05), 2) == round(recall_by_hand, 2))
print("precision (column)=", round(precision_by_hand, 2), " matches precision_score?",
      round(precision_score(y_test, pred_05), 2) == round(precision_by_hand, 2))

## Step 4 · Move the grid — the precision/recall trade-off

Now slide the threshold **down** and redraw. **Predict first:** as the cut drops, which cells grow and which shrink?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
rows = []
for ax, thr in zip(axes, [0.5, 0.3, 0.2]):
    pred = (proba >= thr).astype(int)
    draw_matrix(y_test, pred, ax, f"threshold = {thr}")
    rows.append((thr,
                 int(((pred==1)&(y_test==1)).sum()),   # caught
                 int(((pred==0)&(y_test==1)).sum()),   # missed
                 int(((pred==1)&(y_test==0)).sum()),   # false alarm
                 round(recall_score(y_test, pred), 2),
                 round(precision_score(y_test, pred, zero_division=0), 2)))
plt.tight_layout(); plt.show()

print(f"{"cut":>4}{"caught":>8}{"missed":>8}{"false-alarm":>13}{"recall":>8}{"prec":>7}")
for r in rows:
    print(f"{r[0]:>4}{r[1]:>8}{r[2]:>8}{r[3]:>13}{r[4]:>8}{r[5]:>7}")

**Read it as people.** Dropping the cut from 0.5 to 0.2 catches **3 more frauds** (recall 0.60 → 0.80) for just **1 more false alarm** — an excellent trade. Every number in that table is a person: a robbed customer protected, or an honest customer frozen.

## Step 5 · Choose a threshold for a policy

The fraud team sets a rule: **"catch at least 80% of the fraud."** Find the lowest-false-alarm threshold that meets it, and report what it costs versus the 0.5 default.

In [ ]:
target_recall = 0.80
choice = None
for thr in np.round(np.arange(0.05, 0.95, 0.05), 2):
    pred = (proba >= thr).astype(int)
    if recall_score(y_test, pred) >= target_recall:
        choice = thr   # keep the HIGHEST such threshold (fewest false alarms)

pred_choice = (proba >= choice).astype(int)
fp_choice = int(((pred_choice==1)&(y_test==0)).sum())
fp_05 = int(((pred_05==1)&(y_test==0)).sum())
print("threshold that meets the 80%-recall policy:", choice)
print("recall there:", round(recall_score(y_test, pred_choice), 2),
      " precision:", round(precision_score(y_test, pred_choice), 2))
print("extra false alarms vs 0.5:", fp_choice - fp_05, "  (honest customers newly frozen)")

## Step 6 · ✏️ The recommendation (the module's ethics climax)

Write a short recommendation (**5–6 sentences**) to the bank's fraud team:

- state the **threshold** you'd set and the **recall/precision** it gives;
- map each of the four cells to a **real consequence**;
- name **who benefits and who pays** at your setting;
- acknowledge the strongest objection the **customer-experience team** would raise.

There is no right answer — a strong one names the *people*, not just a preference.

*Your recommendation here:*

## Wrap-up — and the end of Module 3

- The **confusion matrix** shows all four outcomes at once; accuracy, precision, and recall are all read from it.
- **Recall is a row, precision is a column** — read them, don't memorise them.
- The **threshold** slides the whole grid, trading fraud caught against honest customers frozen — with sweet spots and cliffs.
- Choosing where to cut is a **moral decision about who is helped and who is harmed** — the number can't decide it for you.

**The module in one line:** KNN → probabilities → thresholds → precision/recall → the grid — and *evaluation is where ethics lives.*

**Next up (Module 4):** decision trees you can read like rules, and — at last — the **overfitting deep-dive** we've been parking since Session 1.

*Homework: advise a bank's fraud team, in `homework.ipynb` — the module's capstone deliverable.*